In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))


<font size="5" color="red">ch15_데이터베이스 연동</font>

# 1절. SQLite 데이터 베이스 연동
- SQLite 데이터베이스는 별도의 DBMS 없이 SQL을 이용하여 DB엑세스 가능한 디스크 기반 DB
- 가볍게 프로토타입을 만드는 용도로 사용
- 프로젝트 단계 : 분석 - 설계 - 구현 - 테스트 - 고객에게 배포 - 유지보수
-                  SQLite     Oracle/MySQL...
- 파이썬에 자체적으로 내장되어있음. C로 만들어져 속도가 빠름. 단 C라이브러리 필요
- DBbrowser

## 1.1 SQLite 패키지 load

In [3]:
import sqlite3
sqlite3.sqlite_version

'3.40.1'

In [4]:
import pandas as pd
pd.__version__

'1.5.3'

## 1.2 데이터베이스 연결

In [5]:
# DB연결 - 사용 후 반드시 연결 종료를 해야한다
conn = sqlite3.connect('data/ch15_exmple.db')  # 파일이 없으면 빈 새파일을 만든다. 
conn

In [6]:
# 커서객체(SQL문 실행->결과 조회) 생성
cursor = conn.cursor()
cursor

In [7]:
cursor.execute('''
    CREATE TABLE MEMBER(
        NAME TEXT,
        AGE INT,
        EMAIL TEXT
    )
''')

### 1.2.1 insert, update, delete 전송

In [11]:
cursor.execute('''
    INSERT INTO MEMBER VALUES ("홍길동", 20, "hong@nogd.com")
''')
print('수행결과행수 :', cursor.rowcount)

수행결과행수 : 1


In [13]:
sql = 'INSERT INTO MEMBER VALUES ("신길동", 22, "sin@nogd.com")'
cursor.execute(sql)
print('수행결과행수 :', cursor.rowcount)

수행결과행수 : 1


In [14]:
cursor.execute('INSERT INTO MEMBER VALUES ("고길동", 50, "gogog@nogd.com")')
sql = 'INSERT INTO MEMBER VALUES ("마길동", 30, "imma@nogd.com")'
cursor.execute(sql)
print('수행결과행수 :', cursor.rowcount)

수행결과행수 : 1


In [15]:
conn.commit() #(반) conn.rollback() DML에서만 가능

### 1.2.2 select 전송

In [17]:
cursor.execute('''
    SELECT NAME, AGE, AGE+1 NEXTAGE, EMAIL FROM MEMBER
''')

In [20]:
# select 전송
#     SELECT문 실행결과를 받는 함수
#     cursor.fetchone() : 결과를 한 행씩 받을 때 (튜플)
#     cursor.fetchall() : 결과를 전체 받을 때 (튜플 list)
#     cursor.fetchmany(n) : 결과를 n행 받을 때 (튜플 list)
#     cursor.descrption() : header 내용을 포함한 내용들 (list)
print(cursor.fetchone())

None


In [19]:
print(cursor.fetchall())

[]


In [21]:
cursor.execute('''
    SELECT NAME, AGE, AGE+1 NEXTAGE, EMAIL 
        FROM MEMBER
        ORDER BY AGE
''')
# cursor 객체에 위 실행결과가 저장되고, fetch를 이동해 가져오는 것이다
members = cursor.fetchmany(3)
members

[('홍길동', 20, 21, 'hong@nogd.com'),
 ('신길동', 22, 23, 'sin@nogd.com'),
 ('마길동', 30, 31, 'imma@nogd.com')]

In [22]:
member1 = cursor.fetchone()
member1

('고길동', 50, 51, 'gogog@nogd.com')

In [24]:
# 데이터 한줄씩 가져오기
cursor.execute('''
    SELECT NAME, AGE, AGE+1 NEXTAGE, EMAIL 
        FROM MEMBER
        ORDER BY AGE
''')
members = []
while True:
    member = cursor.fetchone()
    if member is None:
        break
    print(member)
    members.append({
        'name' : member[0],
        'age' : member[1],
        'email' : member[2],
    })
pd.DataFrame(members)

('홍길동', 20, 21, 'hong@nogd.com')
('신길동', 22, 23, 'sin@nogd.com')
('마길동', 30, 31, 'imma@nogd.com')
('고길동', 50, 51, 'gogog@nogd.com')


,name,age,email
0,홍길동,20,21
1,신길동,22,23
2,마길동,30,31
3,고길동,50,51


In [31]:
# ★ 가장 많이 사용하는 방법
cursor.execute('''
    SELECT NAME, AGE, AGE+1 NEXTAGE, EMAIL 
        FROM MEMBER
        ORDER BY AGE
''')
members = cursor.fetchall()
df = pd.DataFrame(members, 
                  columns=[header[0] for header in cursor.description])
df

,NAME,AGE,NEXTAGE,EMAIL
0,홍길동,20,21,hong@nogd.com
1,신길동,22,23,sin@nogd.com
2,마길동,30,31,imma@nogd.com
3,고길동,50,51,gogog@nogd.com


In [30]:
# select문을 수행한 필드 정보
cursor.description
[header[0] for header in cursor.description]

['NAME', 'AGE', 'NEXTAGE', 'EMAIL']

## 1.3 SQL구문에 파라미터 사용하기
- qmark(DB에 따라 불가한 경우 있음. Oracle이 대표적)
- named(추천)

In [33]:
# 연결객체 생성 -> 연결객체로 커서 객체 생성 -> 커서객체의 execute함수로 SQL문 전송- > 커서객체의 fetch함수로 select전송
conn = sqlite3.connect('data/ch15_exmple.db')
cursor = conn.cursor()
cursor.execute("SELECT * FROM MEMBER WHERE NAME IN ('홍길동', '고길동')")
cursor.fetchall()

[('홍길동', 20, 'hong@nogd.com'), ('고길동', 50, 'gogog@nogd.com')]

In [34]:
# 파라미터 사용하기 : qmark 이용
conn = sqlite3.connect('data/ch15_exmple.db')
cursor = conn.cursor()
sql = "SELECT * FROM MEMBER WHERE NAME IN (?,?)"
name1 = input('검색할 이름은? ')
name2 = input('검색할 다른 이름은? ')
cursor.execute(sql, (name1, name2) )
cursor.fetchall()

검색할 이름은? 홍길동
검색할 다른 이름은? 성춘향


[('홍길동', 20, 'hong@nogd.com')]

In [35]:
# 파라미터 사용하기 : named 이용
conn = sqlite3.connect('data/ch15_exmple.db')
cursor = conn.cursor()
sql = "SELECT * FROM MEMBER WHERE NAME IN (:name1, :name2)"
name1 = input('검색할 이름은? ')
name2 = input('검색할 다른 이름은? ')
cursor.execute(sql, {'name1': name1, 'name2' : name2} )
cursor.fetchall()

검색할 이름은? 홍길동
검색할 다른 이름은? 성춘향


[('홍길동', 20, 'hong@nogd.com')]

In [37]:
# 파라미터 사용하기 : named 이용 하여 insert 하기
conn = sqlite3.connect('data/ch15_exmple.db')
cursor = conn.cursor()
sql = "INSERT INTO MEMBER VALUES (:name, :age, :email)"
try:
    name = input('이름 : ')
    age = int(input('나이 : '))    
except ValueError:
    print('유효하지 않은 나이를 입력하면 18세로 초기화')
    age=18
finally:
    email = input('이메일 : ')


cursor.execute(sql, {'name': name, 'age' : age, 'email' : email} )
conn.commit()
if cursor.rowcount ==1:
    print('입력성공')
else:
    print('입력실패')


이름 : 박진성
나이 : 삼십
유효하지 않은 나이를 입력하면 18세로 초기화
이메일 : e@e.com
입력성공


In [40]:
# 반드시 종료
cursor.close()
conn.close()

# 2절 오라클 데이터 베이스 연결
- pip install cx_oracle

In [39]:
!pip install cx_oracle

     -------------------------------------- 213.1/213.1 kB 6.5 MB/s eta 0:00:00


In [41]:
import cx_Oracle

In [42]:
# DB 연결객체 생성 방법 1
conn = cx_Oracle.connect('scott', 'tiger', 'localhost:1521/xe')
conn

<cx_Oracle.Connection to scott@localhost:1521/xe>

In [43]:
# DB 연결객체 생성 방법 2
oracle_dsn = cx_Oracle.makedsn(host='localhost', port=1521, sid='xe')
conn = cx_Oracle.connect('scott', 'tiger', dsn = oracle_dsn)
conn

<cx_Oracle.Connection to scott@(DESCRIPTION=(ADDRESS=(PROTOCOL=TCP)(HOST=localhost)(PORT=1521))(CONNECT_DATA=(SID=xe)))>

In [46]:
# 커서 객체 생성
cursor = conn.cursor()
cursor.execute('SELECT EMPNO NO, ENAME, JOB, MGR, SAL, COMM FROM EMP')
emps = cursor.fetchall()
for emp in emps:
    print(emp)

(7369, 'SMITH', 'CLERK', 7902, 800.0, None)
(7499, 'ALLEN', 'SALESMAN', 7698, 1600.0, 300.0)
(7521, 'WARD', 'SALESMAN', 7698, 1250.0, 500.0)
(7566, 'JONES', 'MANAGER', 7839, 2975.0, None)
(7654, 'MARTIN', 'SALESMAN', 7698, 1250.0, 1400.0)
(7698, 'BLAKE', 'MANAGER', 7839, 2850.0, None)
(7782, 'CLARK', 'MANAGER', 7839, 2450.0, None)
(7788, 'SCOTT', 'ANALYST', 7566, 3000.0, None)
(7839, 'KING', 'PRESIDENT', None, 5000.0, None)
(7844, 'TURNER', 'SALESMAN', 7698, 1500.0, 0.0)
(7876, 'ADAMS', 'CLERK', 7788, 1100.0, None)
(7900, 'JAMES', 'CLERK', 7698, 950.0, None)
(7902, 'FORD', 'ANALYST', 7566, 3000.0, None)
(7934, 'MILLER', 'CLERK', 7782, 1300.0, None)


In [47]:
pd.DataFrame(emps,
            columns=[des[0] for des in cursor.description])

,NO,ENAME,JOB,MGR,SAL,COMM
0,7369,SMITH,CLERK,7902.0,800.0,NaN
1,7499,ALLEN,SALESMAN,7698.0,1600.0,300.0
2,7521,WARD,SALESMAN,7698.0,1250.0,500.0
3,7566,JONES,MANAGER,7839.0,2975.0,NaN
4,7654,MARTIN,SALESMAN,7698.0,1250.0,1400.0
5,7698,BLAKE,MANAGER,7839.0,2850.0,NaN
6,7782,CLARK,MANAGER,7839.0,2450.0,NaN
7,7788,SCOTT,ANALYST,7566.0,3000.0,NaN
8,7839,KING,PRESIDENT,NaN,5000.0,NaN
9,7844,TURNER,SALESMAN,7698.0,1500.0,0.0
